# TRIAGE on Kaggle T4×2 (free tier)

Full local pipeline, zero LLM API keys: generator **Llama-3.1-8B-Instruct (4-bit, `llama3.1:8b-instruct-q4_K_M`)** + an LLM judge picked **by index**, all via ollama and all 4-bit quantized (`q4_K_M`). Optional extra judge: `meta/llama-3.3-70b-instruct` on NVIDIA NIM (only when an `NVIDIA_API_KEY` secret is attached).

**The only thing to edit is `JUDGE_MODEL_INDEX` in the EXPERIMENT SETTINGS cell (first code cell).** Everything else (30 test questions, 4-bit weights, memory settings, HCAD evidence settings) is fixed so the three of us produce comparable runs. The choice flows automatically into `.env` â†’ `src/judge_models.py` â†’ every judge call in `src/`; no source edits.

**Before running: notebook Settings →**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (requires phone-verified Kaggle account)

Session budget: **9 h per GPU commit run** (Save & Run All), ~29 GB system RAM. `prepare --rebuild` is the long pole (~75 min on 4 vCPU + T4 embedding). The benchmark checkpoints per question, so an interrupted run resumes safely; checkpoints are tagged with the judge panel and HCAD config, so switching the judge index never mixes scores.

In [ ]:
# 0. EXPERIMENT SETTINGS - the ONLY value a team member edits is JUDGE_MODEL_INDEX.
#    It flows into .env -> src/judge_models.py -> every judge call in src/.
#
#    0 = Qwen3-14B                      (4-bit, ~9.3 GB)
#    1 = DeepSeek-R1-Distill-Qwen-14B   (4-bit, ~9.0 GB)
#    2 = Gemma 3 12B IT                 (4-bit, ~8.2 GB)
#    3 = Qwen2.5-14B-Instruct           (4-bit, ~9.0 GB)
#    4 = Phi-4 14B                      (4-bit, ~9.1 GB)
#    5 = Gemma 2 9B IT                  (4-bit, ~5.8 GB)
#
#    One judge:                  JUDGE_MODEL_INDEX = 0
#    Two-judge consensus panel:  JUDGE_MODEL_INDEX = [0, 3]
JUDGE_MODEL_INDEX = 0

# ---------------------------------------------------------------------------
# FIXED for every team member - keeps runs comparable and inside T4 memory.
# Do NOT edit below this line.
# ---------------------------------------------------------------------------
JUDGE_QUANT = 'q4_K_M'                         # 4-bit weights for every judge
GENERATOR_TAG = 'llama3.1:8b-instruct-q4_K_M'  # 4-bit generator (= src/config.LLM_MODEL)
N_SAMPLES = 30                                 # held-out TEST questions
OLLAMA_KV_CACHE_TYPE = 'q8_0'                  # quantized KV cache (half the memory of f16)
JUDGE_NUM_CTX = 8192                           # pairwise judge prompts need ~4-6k tokens
HCAD_DENSE_ANCHOR_K = 1                        # HCAD evidence assembly (see README)
HCAD_GROUP_FACTS = True

**Judge options (edit `JUDGE_OLLAMA_MODEL` in cell 4):**
- `qwen2.5:14b` (default, recommended): 8b + 14b ≈ 13.7 GB — both fit one T4, no model swapping. Decent judging quality.
- `qwen2.5:7b`: faster pilot-grade fallback if VRAM gets tight.
- `qwen2.5:32b`: does NOT fit a 16 GB T4 (≈19 GB Q4); ollama CPU-offloads and judging crawls (~3–6 tok/s). Not recommended on this hardware — use the A100/Colab Pro or the `JUDGE_PROVIDER=huggingface` cloud hybrid for 32B+ quality.

In [ ]:
# 1. Clone the repo (idempotent: clone once, then require a successful update)
import sys, subprocess, os, json
# Memory guardrails for Kaggle's constrained host RAM.
# These limit CPU-side thread pools; they do NOT shrink the graph or alter results.
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '2')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MALLOC_ARENA_MAX', '2')
print(sys.version)
if sys.version_info >= (3, 13):
    print('Python 3.13 session detected - cell 2 will build a Python 3.12 side-env')

def _diagnose_kill():
    """Print what usually explains a SIGKILL (host out-of-memory)."""
    for cmd in (['free', '-h'], ['nvidia-smi'], ['ollama', 'ps'],
                ['tail', '-n', '30', '/tmp/ollama.log']):
        try:
            subprocess.run(cmd, check=False, timeout=30)
        except Exception as exc:  # diagnostics must never mask the real error
            print(f'(diagnostic {cmd[0]} unavailable: {exc})')

def run_checked(args, label):
    """subprocess.run(check=True) with actionable errors for the common failures."""
    print(f'Running: {label}', flush=True)
    try:
        return subprocess.run(args, check=True)
    except FileNotFoundError as exc:
        raise RuntimeError(f'{label}: executable not found: {args[0]!r}') from exc
    except subprocess.CalledProcessError as exc:
        if exc.returncode in (-9, 137):  # SIGKILL (direct, or via a shell)
            print('\n=== process was SIGKILLed - memory state at failure ===')
            _diagnose_kill()
            raise RuntimeError(
                f'{label} was KILLED by the OS (SIGKILL). On Kaggle this is the '
                'out-of-memory killer: system RAM ran out - typically an ollama '
                'model spilling from GPU into RAM next to the PrimeKG graph. '
                'Fix: use one judge (JUDGE_MODEL_INDEX = <int>) or the smallest '
                'judge (5), re-run the ollama cell, then this cell. Finished '
                'questions resume from the checkpoint.'
            ) from exc
        raise

REPO_URL = 'https://github.com/ajheshbasnet/HCAD.git'
REPO_BRANCH = 'main'
_repo_dir = '/kaggle/working/TRIAGE'   # historical folder name; holds the HCAD clone
_origin = subprocess.run(['git', '-C', _repo_dir, 'remote', 'get-url', 'origin'],
                         capture_output=True, text=True, check=False).stdout.strip()
if not os.path.isdir(f'{_repo_dir}/.git') or _origin.rstrip('/') != REPO_URL:
    if os.path.exists(_repo_dir):
        import shutil
        print(f'Removing stale clone (origin={_origin or "none"})')
        shutil.rmtree(_repo_dir)
    run_checked(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, _repo_dir],
                'clone HCAD')
%cd /kaggle/working/TRIAGE
run_checked(['git', 'pull', '--rebase'], 'update repository')
run_checked(['ls', 'src'], 'verify source tree')
_head_result = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                              capture_output=True, text=True, check=True)
_head = _head_result.stdout.strip()
print('repo HEAD:', _head)
_stale = subprocess.run(
    ['grep', '-q', '"documents", "metadatas", "ids"',
     'src/evaluation/run_base_paper_benchmark.py'], check=False)
if _stale.returncode == 0:
    raise RuntimeError(
        'STALE CLONE: the chromadb ids include bug is still present. '
        'Delete /kaggle/working/TRIAGE and clone the current main branch.'
    )
# Env-pin delivery check: the Oct 25 scispacy failure came from a
# dependency the fresh env build pulled in; the pins must be IN the clone
# or the run is doomed to repeat it.
_pin = subprocess.run(
    ['grep', '-q', 'murmurhash==1.0.12', 'requirements.txt'], check=False)
if _pin.returncode != 0:
    raise RuntimeError(
        'Clone missing the env pins (murmurhash==1.0.12 in '
        'requirements.txt). Delete /kaggle/working/TRIAGE and re-clone '
        'the current main branch.'
    )
print('Env pins present - delivery guard passed.')

In [ ]:
# 2. Install dependencies (10-20 min)
# Python 3.13 sessions build a Python 3.12 side-env; all later stages use PY.
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    run_checked(['bash', '-lc',
                 'set -o pipefail; curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
                 '| tar -xj -C /usr/local bin/micromamba'], 'install micromamba')
    run_checked(['micromamba', 'create', '-y', '-p', '/kaggle/tmp/py312', 'python=3.12'],
                'create Python 3.12 side-env')
    PY = '/kaggle/tmp/py312/bin/python'
run_checked([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
            'install pinned project dependencies')
run_checked([PY, '-m', 'pip', 'install', '-q', 'murmurhash==1.0.12'],
            'pin murmurhash (tokenizer specials regression guard)')
run_checked([PY, '-m', 'pip', 'install', '-q', 'numpy==2.5.4'],
            'install compatible NumPy')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz'],
            'install BC5CDR model')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz'],
            'install JNLPBA model')
print('python:', PY)
run_checked([PY, '-c',
             'import chromadb, spacy, scispacy; '
             'print("stack OK:", chromadb.__version__, spacy.__version__, scispacy.__version__); '
             'from importlib.metadata import version as _v; import numpy; '
             'print("low-level:", _v("murmurhash"), numpy.__version__); '
             'assert chromadb.__version__ == "1.5.9"; '
             'assert spacy.__version__ == "3.8.11"'],
            'verify target Chroma and spaCy versions')


In [ ]:
# 2b. Verify the CUDA runtime used by embedding and tensor training.
import shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No GPU detected in this Kaggle session!\n"
        "Please enable a GPU accelerator in Kaggle Settings:\n"
        "  1. In the right-hand sidebar, click on Notebook options / Settings.\n"
        "  2. Under Accelerator, change from None (CPU) to GPU T4 x 2 (recommended) or GPU P100.\n"
        "  3. Restart the session and re-run."
    )
run_checked(["nvidia-smi"], "verify Kaggle GPUs")
_cuda_check = subprocess.run([PY, "-c", "import torch; print(torch.cuda.is_available())"],
                             capture_output=True, text=True, check=True)
cuda_ok = _cuda_check.stdout.strip().endswith("True")
print("torch.cuda.is_available():", cuda_ok)
if not cuda_ok:
    print("CUDA unavailable - reinstalling torch with CUDA 12.6 runtime...")
    run_checked([PY, "-m", "pip", "install", "-q", "--force-reinstall", "torch",
                 "--index-url", "https://download.pytorch.org/whl/cu126"],
                "install CUDA 12.6 PyTorch")
    _cuda_check = subprocess.run(
        [PY, "-c", "import torch; print(torch.cuda.is_available()); "
         "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"none\")"],
        capture_output=True, text=True, check=True)
    print("after fix:", _cuda_check.stdout.strip())
    if not _cuda_check.stdout.splitlines()[0].strip().endswith("True"):
        raise RuntimeError("CUDA remains unavailable after the PyTorch reinstall.")
else:
    run_checked([PY, "-c", "import torch; print(\"GPU:\", torch.cuda.get_device_name(0))"],
                "verify CUDA device")


In [ ]:
# 3. Install ollama, start it on GPU 1 (or 0 if single-GPU), and verify both pinned models.
run_checked(['apt-get', '-qq', 'install', '-y', 'zstd'], 'install zstd')
run_checked(['bash', '-lc', 'set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'],
            'install ollama')
import os, time, subprocess
# Memory-safe: keep only one Ollama model resident and allow idle unload.
os.environ['OLLAMA_KEEP_ALIVE'] = '5m'
_gpu_res = subprocess.run(['nvidia-smi', '--query-gpu=count', '--format=csv,noheader'], capture_output=True, text=True)
_gpu_count = int(_gpu_res.stdout.strip().splitlines()[0]) if _gpu_res.returncode == 0 and _gpu_res.stdout.strip() else 1
# Keep one model resident at a time. The previous configuration pinned all three
# generator/judge models indefinitely, competing with SciSpacy, PubMedBERT, BM25,
# Chroma and the benchmark process for host RAM/VRAM. Models are still downloaded
# below; Ollama loads them on demand and evicts the previous model as needed.
# Ollama can use both T4s; embedding still uses the notebook's normal CUDA setup.
_ollama_gpu = '0,1' if _gpu_count > 1 else '0'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_QUEUE'] = '1'
print(f'Detected {_gpu_count} GPU(s); assigning Ollama to GPU {_ollama_gpu}')
get_ipython().system_raw(f'CUDA_VISIBLE_DEVICES={_ollama_gpu} ollama serve > /tmp/ollama.log 2>&1 &')

def _ollama_up():
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3):
            return True
    except Exception:
        return False

for _ in range(60):
    if _ollama_up():
        break
    time.sleep(1)
else:
    subprocess.run(['tail', '-n', '40', '/tmp/ollama.log'], check=False)
    raise RuntimeError('ollama serve did not come up within 60 s (log above).')

def pull_with_retry(tag, label, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return run_checked(['ollama', 'pull', tag], f'{label}: {tag} (attempt {attempt}/{attempts})')
        except subprocess.CalledProcessError:
            if attempt < attempts:
                time.sleep(10 * attempt)
    raise RuntimeError(
        f'Could not pull {tag!r}. Check that Internet is ON and that the tag exists: '
        f'https://ollama.com/library/{tag.split(":")[0]}/tags')

pull_with_retry(GENERATOR_TAG, 'pull 4-bit generator')
for _t in JUDGE_TAGS:
    pull_with_retry(_t, 'pull 4-bit judge')
run_checked(['ollama', 'list'], 'verify ollama models')

# --- warm up every model the way src/ calls it, then check GPU residency
def warm_up_and_check_gpu(tag, judge):
    payload = {'model': tag, 'stream': False,
               'messages': [{'role': 'user', 'content': 'Reply with exactly this JSON: {"ok": true}'}],
               'options': {'temperature': 0, 'num_predict': 32}}
    if judge:
        payload['format'] = 'json'
        payload['options']['num_ctx'] = JUDGE_NUM_CTX
    if jm.is_thinking_model(tag):
        payload['think'] = False
    try:
        req = urllib.request.Request('http://localhost:11434/api/chat',
                                     data=json.dumps(payload).encode(),
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=900) as r:
            reply = json.loads(r.read()).get('message', {}).get('content', '')
    except Exception as exc:
        subprocess.run(['tail', '-n', '40', '/tmp/ollama.log'], check=False)
        raise RuntimeError(f'{tag} failed to load/answer: {exc} (ollama log above)') from exc
    if not reply.strip():
        raise RuntimeError(f'{tag} returned an empty reply to the warm-up call.')
    ps = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=False).stdout
    line = next((l for l in ps.splitlines() if l.split() and l.split()[0] == tag), '')
    if not line:
        print(f'WARNING: {tag} not listed by `ollama ps`; GPU residency not verified.')
    elif '100% GPU' not in line:
        raise RuntimeError(
            f'{tag} is NOT fully on the GPU:\n  {line}\nThe CPU part lives in system RAM '
            'next to the PrimeKG graph - this is what gets the benchmark SIGKILLed. Use a '
            'single judge or a smaller judge index.')
    else:
        print(f'{tag}: warm-up OK, 100% GPU')

warm_up_and_check_gpu(GENERATOR_TAG, judge=False)
for _t in JUDGE_TAGS:
    warm_up_and_check_gpu(_t, judge=True)
run_checked(['ollama', 'ps'], 'show resident models')

In [ ]:
# 4. Environment file - all API keys go HERE (this is the only cell with keys).
#    Preferred: Notebook editor -> Add-ons -> Secrets -> attach secrets named
#    WANDB_API_KEY and TYPESAFE_API_KEY. Secrets never appear in the notebook
#    code or its saved versions. The literals below are a fallback for
#    non-Kaggle machines only - leave them empty if you use Secrets.
#    - WANDB_API_KEY: telemetry + the off-box checkpoint backup artifact
#      (uploaded every 5 questions; the only copy that survives a killed
#      session). Without it W&B stays disabled and you rely on /kaggle/working.
#    - TYPESAFE_API_KEY: optional, live JEV Decision Gate. Without it the
#      gate runs in local calibrated mode (what the current run uses).
WANDB_API_KEY = ""      # fallback literal only
TYPESAFE_API_KEY = ""   # fallback literal only
NVIDIA_API_KEY = ""     # optional: enables the third (Llama-3.3-70B) judge

def _secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return ""

WANDB_API_KEY = _secret("WANDB_API_KEY") or WANDB_API_KEY
TYPESAFE_API_KEY = _secret("TYPESAFE_API_KEY") or TYPESAFE_API_KEY
NVIDIA_API_KEY = _secret("NVIDIA_API_KEY") or NVIDIA_API_KEY

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    # Judge panel from the EXPERIMENT SETTINGS cell; src/judge_models.py maps
    # the index(es) to the exact 4-bit ollama tags. Judges are strict: never swapped.
    f.write(f"JUDGE_MODEL_INDEX={','.join(map(str, JUDGE_INDICES))}\n")
    f.write(f"JUDGE_QUANT={JUDGE_QUANT}\n")
    f.write(f"JUDGE_NUM_CTX={JUDGE_NUM_CTX}\n")
    if NVIDIA_API_KEY:
        f.write(f"NVIDIA_API_KEY={NVIDIA_API_KEY}\n")
        f.write("JUDGE_ENABLE_API_JUDGE=1\n")   # + Llama-3.3-70B via NIM
    if WANDB_API_KEY:
        f.write(f"WANDB_API_KEY={WANDB_API_KEY}\n")
    else:
        f.write("WANDB_MODE=disabled\n")
    if TYPESAFE_API_KEY:
        f.write(f"TYPESAFE_API_KEY={TYPESAFE_API_KEY}\n")
# Print keys only (redacted values) so the saved notebook log stays clean.
print('\n'.join(line.split('=')[0] + '=***' for line in open('.env').read().strip().split('\n') if line.strip()))

# The kernel environment is inherited by every src/ subprocess and WINS over
# .env (load_dotenv(override=False)), so mirror the selection here and drop
# stale values from an earlier selection in this session.
for _k in ('JUDGE_ENSEMBLE', 'JUDGE_OLLAMA_MODEL'):
    os.environ.pop(_k, None)
os.environ['JUDGE_MODEL_INDEX'] = ','.join(map(str, JUDGE_INDICES))
os.environ['JUDGE_QUANT'] = JUDGE_QUANT
os.environ['JUDGE_NUM_CTX'] = str(JUDGE_NUM_CTX)

# Prove src/ resolves exactly the models this notebook pulled and verified.
_resolved = subprocess.run(
    [PY, '-c', 'import json; from src.config import judge_ensemble_specs, LLM_MODEL; '
               'print(json.dumps({"judges": [m for p, m in judge_ensemble_specs() if p == "ollama"], '
               '"generator": LLM_MODEL}))'],
    capture_output=True, text=True)
if _resolved.returncode != 0:
    raise RuntimeError('src.config failed to resolve the judges:\n' + _resolved.stderr[-2000:])
_resolved_lines = _resolved.stdout.strip().splitlines()
if not _resolved_lines:
    raise RuntimeError('src.config printed nothing:\n' + _resolved.stderr[-2000:])
_resolved = json.loads(_resolved_lines[-1])
if _resolved['judges'] != JUDGE_TAGS:
    raise RuntimeError(f"src/ resolves judges {_resolved['judges']}, notebook pulled {JUDGE_TAGS}.")
if _resolved['generator'] != GENERATOR_TAG:
    raise RuntimeError(f"src/ generator is {_resolved['generator']!r}, notebook pulled {GENERATOR_TAG!r}.")
EXPECTED_JUDGES = [f'ollama:{t}' for t in JUDGE_TAGS]
print('src/ judges   :', EXPECTED_JUDGES)
print('src/ generator:', _resolved['generator'])

### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [ ]:
# 4b. RESTORE (only when the artifacts dataset is attached).
# Restore by archive, extracted store, or legacy flat layout; verify all required files.
RESTORE = True
RESTORED = False
if RESTORE:
    import glob, os, shutil
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/vectordb', exist_ok=True)
    tarballs = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
                      + glob.glob('/kaggle/input/**/*.tgz', recursive=True))
    zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
    if tarballs:
        print(f'Restoring from {tarballs[0]} ...')
        run_checked(['tar', 'xzf', tarballs[0]], 'extract restore archive')
    elif zips:
        print(f'Restoring from {zips[0]} ...')
        run_checked(['unzip', '-q', '-o', zips[0]], 'extract zip archive')
    else:
        chroma_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if chroma_hits:
            src_vdb = os.path.dirname(chroma_hits[0])
            print(f'Restoring extracted store from {src_vdb} ...')
            shutil.copytree(src_vdb, 'data/vectordb', dirs_exist_ok=True)
            for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss',
                          'sapbert_node_names.json'):
                hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if hits:
                    shutil.copy2(hits[0], f'data/processed/{fname}')
                    print(f'  + {fname}')
        else:
            legacy = '/kaggle/input/triage-artifacts'
            if os.path.isdir(legacy):
                shutil.copytree(legacy, 'data/processed', dirs_exist_ok=True)
                legacy_vdb = os.path.join(legacy, 'vectordb')
                if os.path.isdir(legacy_vdb):
                    shutil.copytree(legacy_vdb, 'data/vectordb', dirs_exist_ok=True)

    # Check if files were extracted or mounted anywhere under /kaggle/input and copy if missing
    chroma = 'data/vectordb/chroma.sqlite3'
    gpickle = 'data/processed/primekg_graph.gpickle'
    index = 'data/processed/sapbert_node_index.faiss'
    if not (os.path.exists(chroma) and os.path.exists(gpickle) and os.path.exists(index)):
        c_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if c_hits and not os.path.exists(chroma):
            shutil.copytree(os.path.dirname(c_hits[0]), 'data/vectordb', dirs_exist_ok=True)
        for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss', 'sapbert_node_names.json'):
            target = f'data/processed/{fname}'
            if not os.path.exists(target):
                f_hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if f_hits:
                    shutil.copy2(f_hits[0], target)

    if (os.path.exists(chroma) and os.path.getsize(chroma) > 10_000_000
            and os.path.exists(gpickle) and os.path.exists(index)):
        RESTORED = True
        print(f'Restore OK: chroma.sqlite3 = {os.path.getsize(chroma)/1e6:.0f} MB')
        run_checked(['ls', '-la', 'data/processed/', 'data/vectordb/'],
                    'verify restored artifacts')
    else:
        print('No complete pre-built artifacts found in /kaggle/input (or restore was skipped).')
        print('Listing of /kaggle/input:')
        subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], check=False)
        print('RESTORED = False. Stage 6 (cell 9) will prepare the knowledge graph and vector store from scratch.')
        RESTORED = False
        run_checked(['echo', 'Artifact restore skipped; cell 6 will build from scratch.'],
                    'log restore skip')


In [ ]:
# 5. Full test suite (run only in this pinned Kaggle environment).
run_checked([PY, '-m', 'pytest', 'tests', '-q'], 'full pytest suite')


### Memory-safety change

The prepare stage is the known peak-RAM operation. This revision records host RAM/disk/GPU
status, limits CPU thread pools, preserves the full data scope, and refuses to launch another
full rebuild unless you explicitly set `ALLOW_FULL_REBUILD = True` in cell 6. The preferred
recovery is to attach a previously saved `triage_artifacts.tar.gz` as a Kaggle Input Dataset
and restore it in cell 4b. Reducing the PMC article count would change the experiment, so this
patch does not do that.


In [ ]:
# 6. Prepare the knowledge graph and vector store (the multi-hour, peak-RAM stage).
# Prefer RESTORE=True in cell 4b and attach the saved artifacts Kaggle Dataset.
# A full rebuild is intentionally opt-in because this is the stage that OOMed.
import os, shutil, subprocess, time, json

if RESTORED:
    print('Skipped prepare: complete artifacts restored in cell 4b.')
    run_checked(['du', '-sh', 'data/vectordb', 'data/processed'],
                'verify restored store sizes')
else:
    # Preflight: record host RAM and free disk before starting the expensive child process.
    def _meminfo_mb():
        vals = {}
        try:
            for line in open('/proc/meminfo'):
                k, v = line.split(':', 1)
                if k in ('MemTotal', 'MemAvailable'):
                    vals[k] = int(v.strip().split()[0]) // 1024
        except OSError:
            pass
        return vals

    _mi = _meminfo_mb()
    _disk = shutil.disk_usage('/kaggle/working')
    print('Host memory (MB):', _mi)
    print('Free working-disk (GB):', round(_disk.free / 1e9, 2))
    print('GPU status:')
    subprocess.run(['nvidia-smi'], check=False)

    # Keep the full PrimeKG/PMC scope. Do not lower PMC_NUM_ARTICLES to hide OOM:
    # that would change the dataset and make results incomparable.
    os.environ['OMP_NUM_THREADS'] = '2'
    os.environ['OPENBLAS_NUM_THREADS'] = '2'
    os.environ['MKL_NUM_THREADS'] = '2'
    os.environ['NUMEXPR_NUM_THREADS'] = '2'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['MALLOC_ARENA_MAX'] = '2'
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ.setdefault('PMC_NUM_ARTICLES', '1000')

    ALLOW_FULL_REBUILD = False  # Set True only if you intentionally want to retry the peak-RAM build.
    if not ALLOW_FULL_REBUILD:
        raise RuntimeError(
            'No complete prepared artifacts were restored. Full prepare previously OOMed on this '
            'Kaggle session. Do not rerun blindly: attach the saved triage_artifacts.tar.gz (or '
            'the extracted data/processed + data/vectordb) as a Kaggle Input Dataset, keep RESTORE=True '
            'in cell 4b, then rerun from cell 4b. If this is your first-ever build, set '
            'ALLOW_FULL_REBUILD=True below only after accepting the RAM risk.'
        )

    _prepare_log = '/kaggle/working/prepare.log'
    print('Starting full prepare; log:', _prepare_log)
    with open(_prepare_log, 'w', encoding='utf-8') as _log:
        _proc = subprocess.Popen(
            [PY, 'src/main.py', 'prepare', '--rebuild'],
            stdout=_log, stderr=subprocess.STDOUT
        )
        _rc = _proc.wait()
    if _rc != 0:
        print('Prepare failed. Last 100 log lines:')
        subprocess.run(['tail', '-n', '100', _prepare_log], check=False)
        raise subprocess.CalledProcessError(_rc, [PY, 'src/main.py', 'prepare', '--rebuild'])
    for required in ('data/vectordb/chroma.sqlite3',
                     'data/processed/primekg_graph.gpickle',
                     'data/processed/sapbert_node_index.faiss'):
        if not os.path.isfile(required) or os.path.getsize(required) == 0:
            raise RuntimeError(f'Prepare completed without required output: {required}')


In [ ]:
# 6b. Snapshot prepared artifacts before the benchmark.
_archive = '/kaggle/working/triage_artifacts.tar.gz'
run_checked(['tar', 'czf', _archive,
             'data/processed/primekg_graph.gpickle',
             'data/processed/sapbert_node_index.faiss',
             'data/processed/sapbert_node_names.json', 'data/vectordb'],
            'snapshot prepared artifacts')
if not os.path.isfile(_archive) or os.path.getsize(_archive) <= 10_000_000:
    raise RuntimeError('Artifact archive is missing or unexpectedly small.')
run_checked(['du', '-sh', _archive], 'verify artifact archive')


In [ ]:
# 7. Smoke test one query through all three systems.
run_checked([PY, 'src/main.py', 'demo'], 'run three-system smoke test')


In [ ]:
# 8. Optional session-1 pilot.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_PILOT = False
if RUN_PILOT:
    _pilot_path = 'results/base_paper_benchmark_results.json'
    _pilot_before = os.stat(_pilot_path).st_mtime_ns if os.path.exists(_pilot_path) else None
    run_checked([PY, 'src/main.py', 'base-paper', '--samples', '24', '--plot'],
                'run 24-question pilot')
    if not os.path.exists(_pilot_path) or (
            _pilot_before is not None and os.stat(_pilot_path).st_mtime_ns <= _pilot_before):
        raise RuntimeError('Pilot exited without producing a fresh results file.')
else:
    print('Skipped pilot (RUN_PILOT=False).')


In [ ]:
# 9a. Train the learned query-adaptive tensor on the dev pool.
TRAIN_TENSOR = True  # Re-enabled: Phase-2 listwise ranking loss and SciSpacy tokenizer recovery operational
if TRAIN_TENSOR:
    _tensor_path = 'models/dynamic_tensor.pt'
    _tensor_before = os.stat(_tensor_path).st_mtime_ns if os.path.exists(_tensor_path) else None
    run_checked([PY, 'src/main.py', 'train-tensor', '--samples', '60', '--epochs', '3'],
                'train dev-pool tensor')
    if not os.path.exists(_tensor_path) or (
            _tensor_before is not None and os.stat(_tensor_path).st_mtime_ns <= _tensor_before):
        raise RuntimeError('Tensor training exited without producing a fresh checkpoint.')
else:
    print('Skipped tensor training (TRAIN_TENSOR=False).')


In [ ]:
# 9c. Deliverable-B dev-pool retrieval eval (Phase 3).
# Paired before/after of channel-presence masking + relation-vocab
# mapping + hybrid dense/structured scoring + Gate-2 saturation cap on
# the DEV pool (retrieval-only, no judge). Calibrates tau_sat (p25 of
# gold dev candidate scores, hybrid scale) and prints the GO/STOP
# verdict (GO unless NDCG@5 drops vs the legacy baseline arm).
# The test pool stays untouched until GO.
import glob, json, os
run_checked([PY, 'src/main.py', 'retrieval-eval', '--samples', '60', '--pool', 'dev'],
            'run Deliverable-B dev-pool retrieval eval')
_eval_files = sorted(glob.glob('results/retrieval_eval_dev_n*.json'))
if not _eval_files:
    raise RuntimeError('Retrieval eval exited without producing results.')
with open(_eval_files[-1], encoding='utf-8') as _f:
    _retrieval_eval = json.load(_f)
if _retrieval_eval.get('go_stop', {}).get('decision') != 'GO':
    print(
        'NOTICE: Deliverable-B GO/STOP gate returned '
        + str(_retrieval_eval.get('go_stop')))
else:
    print('Deliverable-B GO verdict:',
          _retrieval_eval.get('delta_vs_baseline'))


In [ ]:
# 9d. Phase-4a hub-weighting ablation (dev pool, retrieval-only).
# Compares desc (legacy hub-favoring) vs soft (deployed: inverse
# log-degree, NO hard cap) vs asc_cap (hard deg<=cap drop, expected
# to lose canonical receptors like NR3C1). Gold Recall@5/NDCG@5 plus
# canonical-hub retention rate.
run_checked([PY, 'src/main.py', 'hub-eval', '--samples', '60', '--pool', 'dev'],
            'run Phase-4a hub ablation')
import glob as _glob, json as _json
_hub_files = sorted(_glob.glob('results/hub_ablation_dev_n*.json'))
if _hub_files:
    with open(_hub_files[-1], encoding='utf-8') as _f:
        print(_json.dumps(_json.load(_f), indent=2))


In [ ]:
# 9. Main full benchmark: static weights and held-out test questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
print('HCAD_EXPECTED_GATE_MODE:', os.environ['HCAD_EXPECTED_GATE_MODE'])
# Deliverable-B: activate the calibrated Gate-2 saturation cap.
import glob as _glob, json as _json
_retrieval_eval_files = sorted(_glob.glob('results/retrieval_eval_dev_n*.json'))
if _retrieval_eval_files:
    with open(_retrieval_eval_files[-1], encoding='utf-8') as _f:
        _tau_sat = _json.load(_f).get('tau_sat')
    if _tau_sat:
        os.environ['HCAD_TAU_SAT'] = str(_tau_sat)
        print('Gate-2 saturation cap active: tau_sat =', _tau_sat)

RUN_FULL_BENCHMARK = True
# Arms: closed-book Llama-3.1-8B, BM25, dense S-PubMedBERT, hybrid RRF, HCAD-RAG.
# RUN_ABLATIONS adds w/o Decoupling / w/o Soft Gram-Schmidt / w/o Decision Gate
# (~+50% judge calls; on T4x2 run it in its own session if time is short).
RUN_ABLATIONS = False
if RUN_FULL_BENCHMARK:
    _results_path = 'results/base_paper_benchmark_results.json'
    _results_before = os.stat(_results_path).st_mtime_ns if os.path.exists(_results_path) else None
    _bench_cmd = [PY, 'src/main.py', 'base-paper', '--samples', str(N_SAMPLES), '--plot']
    if RUN_ABLATIONS:
        _bench_cmd.append('--ablations')
    # Capture host-memory headroom while the benchmark runs. This does not change
    # the benchmark samples, arms, judges, or scoring protocol.
    import subprocess as _sp, time as _time
    _proc = _sp.Popen(_bench_cmd)
    while _proc.poll() is None:
        try:
            _mem = {}
            with open('/proc/meminfo') as _mf:
                for _line in _mf:
                    if _line.startswith(('MemAvailable:', 'MemTotal:')):
                        _k, _v = _line.split(':', 1)
                        _mem[_k] = int(_v.strip().split()[0]) // 1024
            _rss = _sp.run(['ps', '-o', 'rss=', '-p', str(_proc.pid)],
                           capture_output=True, text=True).stdout.strip() or 'n/a'
            _ollama = _sp.run(['bash', '-lc', "ps -C ollama -o rss= | awk '{s+=$1} END {print (s+0)/1024}'"],
                              capture_output=True, text=True).stdout.strip() or 'n/a'
            print(f"[RAM] available={_mem.get('MemAvailable','?')} MB / total={_mem.get('MemTotal','?')} MB; benchmark RSS={_rss} KB; Ollama RSS={_ollama} MB", flush=True)
        except Exception as _diag_err:
            print('[RAM] diagnostic unavailable:', _diag_err, flush=True)
        _time.sleep(20)
    if _proc.returncode != 0:
        raise _sp.CalledProcessError(_proc.returncode, _bench_cmd)
    if not os.path.exists(_results_path) or (
            _results_before is not None and os.stat(_results_path).st_mtime_ns <= _results_before):
        raise RuntimeError('Benchmark exited without producing fresh results.')
    with open(_results_path, encoding='utf-8') as _f:
        _benchmark_results = json.load(_f)
        _run_manifest = _benchmark_results.get('run_manifest', {})
    if (_benchmark_results.get('num_samples') != N_SAMPLES
            or _run_manifest.get('question_pool') != 'test'):
        raise RuntimeError('Benchmark result count or question pool does not match this run.')
    if _benchmark_results.get('sample_source_counts') != {'paper_protocol': N_SAMPLES}:
        raise RuntimeError(f'Benchmark results do not contain all {N_SAMPLES} paper-protocol questions.')
    for _key in ('statistical_tests', 'inter_judge_agreement', 'latency_breakdown'):
        if not _benchmark_results.get(_key):
            raise RuntimeError(f'Benchmark results are missing {_key}.')
    _ran_judges = [j.get('name') for j in _run_manifest.get('judges', [])]
    if _ran_judges[:len(EXPECTED_JUDGES)] != EXPECTED_JUDGES:
        raise RuntimeError(f'Results were scored by {_ran_judges}, expected {EXPECTED_JUDGES}.')
    if not os.path.exists('results/latency_breakdown.json'):
        raise RuntimeError('results/latency_breakdown.json was not written.')
else:
    print('Skipped full benchmark (RUN_FULL_BENCHMARK=False).')

In [ ]:
# 9b. Optional learned-tensor ablation on the identical TEST questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_ABLATION = False
if RUN_ABLATION:
    import os
    os.environ['HCAD_TENSOR_MODE'] = 'learned'
    os.environ['BENCHMARK_CHECKPOINT'] = 'base_paper_checkpoint_learned.jsonl'
    os.environ['BENCHMARK_RESULTS_FILE'] = 'base_paper_benchmark_results_learned.json'
    _ablation_path = 'results/base_paper_benchmark_results_learned.json'
    _ablation_before = os.stat(_ablation_path).st_mtime_ns if os.path.exists(_ablation_path) else None
    try:
        run_checked([PY, 'src/main.py', 'base-paper', '--samples', str(N_SAMPLES), '--plot'],
                    'run learned-tensor ablation')
        if not os.path.exists(_ablation_path) or (
                _ablation_before is not None and os.stat(_ablation_path).st_mtime_ns <= _ablation_before):
            raise RuntimeError('Ablation exited without producing fresh results.')
    finally:
        del os.environ['HCAD_TENSOR_MODE'], os.environ['BENCHMARK_CHECKPOINT'], os.environ['BENCHMARK_RESULTS_FILE']
else:
    print('Skipped learned-tensor ablation (RUN_ABLATION=False).')

In [ ]:
# 10. Inspect the fresh benchmark results.
import json
with open('results/base_paper_benchmark_results.json', encoding='utf-8') as f:
    r = json.load(f)
if r.get('num_samples') != N_SAMPLES or r.get('run_manifest', {}).get('question_pool') != 'test':
    raise RuntimeError(f'Results are stale or do not contain the {N_SAMPLES}-question test pool.')
print('judges:', [j.get('name') for j in r['run_manifest'].get('judges', [])])
print('HCAD evidence config:', r['run_manifest'].get('hcad_evidence_config'))
# What each arm actually gave the generator: if HCAD's gold_in_evidence_rate or
# mean_words trails dense RAG, the gap is evidence SELECTION, not the prompt.
print('evidence diagnostics:', json.dumps(r.get('evidence_diagnostics'), indent=2))
print(json.dumps({k: r.get(k) for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'judge_failures')}, indent=2)[:2500])
for _key, _cmp in r['statistical_tests']['comparisons'].items():
    _c = _cmp['consensus']
    if _c.get('n_paired'):
        _ci = _c['paired_difference_ci']
        print(f"{_key:28s} diff={_ci['mean_difference']:+.2f} "
              f"[{_ci['ci_low']:+.2f}, {_ci['ci_high']:+.2f}] "
              f"Wilcoxon p={_c['wilcoxon_signed_rank']['p_value']:.4f} "
              f"McNemar p={_c['mcnemar_exact']['p_value']:.4f}")
print(json.dumps(r['inter_judge_agreement'].get('fleiss'), indent=2)[:1500])

In [ ]:
# 10b. Paired-polarity benchmark (indication vs contraindication, retrieval-only).
import os, json
_pol_path = 'results/polarity_benchmark.json'
_pol_before = os.stat(_pol_path).st_mtime_ns if os.path.exists(_pol_path) else None
run_checked([PY, 'src/main.py', 'polarity', '--arms', 'bm25,dense,hybrid,hcad,hcad_kg'],
            'run paired-polarity benchmark')
if not os.path.exists(_pol_path) or (
        _pol_before is not None and os.stat(_pol_path).st_mtime_ns <= _pol_before):
    raise RuntimeError('Polarity benchmark exited without producing fresh results.')
_pol = json.load(open(_pol_path, encoding='utf-8'))
for _arm, _res in _pol['arms'].items():
    _o = _res['overall']
    print(f"{_arm:8s} PolarityRecall@{_pol['k']}={_o['polarity_recall_at_k']['mean']:.3f} "
          f"PolarityErrorRate={_o['polarity_error_rate']['mean']:.3f}")

In [ ]:
# 10c. MedHallu: classifier-only detector mode + HCAD-RAG generated answers.
import os
MEDHALLU_N = 200
for _system in ('detector', 'hcad'):
    run_checked([PY, 'src/main.py', 'medhallu', '--system', _system, '--n', str(MEDHALLU_N)],
                f'run MedHallu ({_system})')
import glob
_hallu = sorted(glob.glob('results/hallucination/medhallu_*.json'))
if len(_hallu) < 2:
    raise RuntimeError(f'MedHallu outputs missing: {_hallu}')
print('MedHallu outputs:', _hallu)

In [ ]:
# 10d. Tables + figures (incl. 5-stage methodology diagram) from the saved JSON.
#      Fails if any table number disagrees with the results JSON.
run_checked([PY, 'src/main.py', 'figures'], 'generate tables and figures')
run_checked(['ls', '-la', 'results/figures', 'results/tables'], 'list figures and tables')

In [ ]:
# 11. Optional live API check with checked HTTP status and clean shutdown.
import subprocess
_server_log = open('/tmp/serve.log', 'w')
_server = subprocess.Popen(
    [PY, 'src/main.py', 'serve', '--host', '127.0.0.1', '--port', '8000'],
    stdout=_server_log, stderr=subprocess.STDOUT,
)
try:
    run_checked(['curl', '--fail', '--show-error', '--retry', '30',
                 '--retry-connrefused', '--retry-delay', '2',
                 'http://127.0.0.1:8000/api/triage/health'],
                'check API health endpoint')
    run_checked(['curl', '--fail', '--show-error', '-X', 'POST',
                 'http://127.0.0.1:8000/api/triage/query',
                 '-H', 'Content-Type: application/json',
                 '-d', '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' ],
                'check clinical query endpoint')
finally:
    _server.terminate()
    _server.wait(timeout=10)
    _server_log.close()


In [ ]:
# 12. Copy results and resume files to the working directory.
import glob, os, shutil
_copy_paths = [
    'results/base_paper_checkpoint.jsonl',
    'results/base_paper_benchmark_results.json',
    'results/base_paper_checkpoint_learned.jsonl',
    'results/base_paper_benchmark_results_learned.json',
    'models/dynamic_tensor.pt',
    'results/latency_breakdown.json',
    'results/polarity_benchmark.json',
    'results/polarity_benchmark_per_query.jsonl',
]
_copy_paths.extend(glob.glob('results/paper_questions_*.json'))
_copy_paths.extend(glob.glob('results/hallucination/medhallu_*'))
for _dir in ('results/figures', 'results/tables'):
    if os.path.isdir(_dir):
        shutil.copytree(_dir, os.path.join('/kaggle/working', os.path.basename(_dir)),
                        dirs_exist_ok=True)
        print('Copied dir:', _dir)
for _path in _copy_paths:
    if os.path.isfile(_path):
        shutil.copy2(_path, '/kaggle/working/')
        print('Copied:', _path)
    elif _path in ('results/base_paper_checkpoint.jsonl',
                   'results/base_paper_benchmark_results.json'):
        raise RuntimeError(f'Required benchmark output is missing: {_path}')
run_checked(['ls', '-la', '/kaggle/working/'], 'verify saved outputs')


## Troubleshooting

- **OOM in cell 6:** Kaggle gives ~29 GB RAM; the PrimeKG graph build is the peak. If it dies, there is no swap — restart and restore from the artifacts dataset instead of re-running prepare.
- **Judge model:** default `qwen2.5:14b` fits alongside the 8b generator on one T4. `qwen2.5:7b` if tight; `qwen2.5:32b` does not fit a T4 (heavy CPU offload, very slow) — for 32B+ quality use the `JUDGE_PROVIDER=huggingface` cloud hybrid instead.
- **ollama dead after restart:** re-run cell 3 (the server doesn't survive kernel restarts).
- **Checkpoint reuse:** re-running cell 9 with the same sample count skips already-evaluated questions and aggregates over the full set.
- **9 h GPU commit cap:** prepare + pilot + full benchmark together can exceed it. That is why cell 9 is gated by RUN_FULL_BENCHMARK - do prepare + pilot in session 1, full benchmark in session 2 (restore first, cell 4b).
- **Session killed mid-benchmark:** the checkpoint (`results/base_paper_checkpoint.jsonl`) and question cache are auto-uploaded to W&B as an artifact (`benchmark-checkpoint`, every 5 questions) as long as `WANDB_API_KEY` is set. Download the latest artifact version from the `hcad-rag` W&B project, drop the two files into `results/`, restore artifacts (cell 4b), and re-run cell 9 - it resumes exactly where it died.
